# Feature Engineering

In [0]:
# Create schema to RetailCast data
spark.sql("""
CREATE SCHEMA IF NOT EXISTS retailcast_solo.retailcast_solo_ML_features
""")

DataFrame[]

In [0]:
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import col, to_timestamp, unix_timestamp, when, round, lag, mean

# ======================================================
# Inicializar Spark
# ======================================================

spark = SparkSession.builder \
    .appName("Feature Table for AutoML - RetailCast") \
    .getOrCreate()

# Carregar camada Gold (Rich Layer)
df = spark.sql("SELECT * FROM retailcast_solo.retailcast_solo_gold.rc_aggregation_gold")

# ======================================================
# Criação de novas features
# . Ao criar novas versões, tornar a versão anterior em comentário, para registo do histórico e rastreabilidade.
# ======================================================

# 1. Remover zeros estruturais (lojas fechadas)
df = df.filter(col("LOJA_ABERTA") == 1)

# 2. Garantir cast correto para funções de janela
df = df.withColumn("DATA", col("DATA").cast("date"))

# 3. Janelas temporais rigorosas (excluem D-0 para evitar data leakage)[cite: 2]
window_lags = Window.partitionBy("FK_LOJA").orderBy("DATA")
window_roll_1 = Window.partitionBy("FK_LOJA").orderBy("DATA").rowsBetween(-1, -1) # idêntico ao lag(1)
window_roll_7 = Window.partitionBy("FK_LOJA").orderBy("DATA").rowsBetween(-7, -1)
window_roll_14 = Window.partitionBy("FK_LOJA").orderBy("DATA").rowsBetween(-14, -1)

# 4: Expansiva por Região (do primeiro registo até D-1)
window_regiao_historica = Window.partitionBy("REGIAO").orderBy("DATA").rowsBetween(Window.unboundedPreceding, -1)

# 5. Computação de Lags, Averages e Interações[cite: 2]
##  . Lag - calcula média de vendas, dentro de um horizonte temporal definido.
##  . Roll Avg - calcula média móvel de vendas, também dentro de um horiz. temporal definido.

df_features = df.withColumn(
    "LAG_1_VENDAS", lag("TARGET_VENDAS_DIARIAS_EUR", 1).over(window_lags)
).withColumn(
    "LAG_7_VENDAS", lag("TARGET_VENDAS_DIARIAS_EUR", 7).over(window_lags)
).withColumn(
    "LAG_14_VENDAS", lag("TARGET_VENDAS_DIARIAS_EUR", 14).over(window_lags)
).withColumn(
    "ROLL_AVG_1_VENDAS", mean("TARGET_VENDAS_DIARIAS_EUR").over(window_roll_1)
).withColumn(
    "ROLL_AVG_7_VENDAS", mean("TARGET_VENDAS_DIARIAS_EUR").over(window_roll_7)
).withColumn(
    "ROLL_AVG_14_VENDAS", mean("TARGET_VENDAS_DIARIAS_EUR").over(window_roll_14)
    ### ====================  NÃO ALTERAR AS FEATURES ACIMA  ====================
).withColumn(
    "IS_WEEKEND",
    when(col("DIA_SEMANA").isin(6, 7), 1).otherwise(0)
).withColumn(
    "EVENTO_FIM_SEMANA",
    col("IS_WEEKEND") * col("IS_EVENT")
).withColumn("IS_REG_LIS_NOR", when(col("REGIAO").isin("Lisboa Norte"), 1).otherwise(0)

)

# ========== HISTORICO DE FEATURES CRIADAS
#.withColumn("TOP4_REGIÃO", when(col("REGIAO").isin("Região Norte", "Porto", "Lisboa Norte", "Margem Sul"), 1).otherwise(0))
# withColumn("IS_REG_NORTE", when(col("REGIAO").isin("Região Norte"), 1).otherwise(0))
# withColumn("IS_REG_PORTO", when(col("REGIAO").isin("Porto"), 1).otherwise(0))
# withColumn("IS_REG_LIS_NOR", when(col("REGIAO").isin("Lisboa Norte"), 1).otherwise(0))
# .withColumn("TAMANHO_LOJA",
#    when(col("TOTAL_SKUS_LOJA") < 5500, "4") #Pequena
#    .when(col("TOTAL_SKUS_LOJA") < 12500, "3") #Media
#    .when(col("TOTAL_SKUS_LOJA") < 27000, "2") #Grande
#    .otherwise("1") #Hiper)
##).withColumn(
#    "IS_LOJA_PEQUENA", when(col("TOTAL_SKUS_LOJA") < 5500, 1).otherwise(0)
#).withColumn(
#     "IS_LOJA_MEDIA", when((col("TOTAL_SKUS_LOJA") >= 5500) & (col("TOTAL_SKUS_LOJA") < 12500), 1).otherwise(0)
#).withColumn(
#     "IS_LOJA_GRANDE", when((col("TOTAL_SKUS_LOJA") >= 12500) & (col("TOTAL_SKUS_LOJA") < 27000), 1).otherwise(0)
#).withColumn(
#     "IS_HIPERMERCADO", when(col("TOTAL_SKUS_LOJA") >= 27000, 1).otherwise(0)


# 6. Limpar nulos iniciais das janelas
df_features = df_features.dropna(subset=["LAG_14_VENDAS", "ROLL_AVG_14_VENDAS"])

# ======================================================
# Seleção Final das Features
# ======================================================

features_df = df_features.select(
    "TARGET_VENDAS_DIARIAS_EUR",
    "DATA",
    "PRODUTIVIDADE/HORA",
    "N_COLABORADORES",
    "SELF_CHECKOUTS",
    "LAG_1_VENDAS",
    "LAG_7_VENDAS",
    "LAG_14_VENDAS",
    "ROLL_AVG_1_VENDAS",
    "ROLL_AVG_7_VENDAS",
    "ROLL_AVG_14_VENDAS",
    "CAIXAS_TRADICIONAIS",
    "SKUS_+",
    "IS_EVENT",
    "EVENTO_FIM_SEMANA",
    "IS_REG_LIS_NOR", ### não alterar as features a partir daqui (baseline)
    "LOJA"
    )


#   . Agora faz overwrite e salva sempre como V2.
features_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("retailcast_solo.retailcast_solo_ML_features.rc_ML_features")

print("✅ Feature table criada com sucesso para AutoML.")


✅ Feature table criada com sucesso para AutoML.
